# Fase C — Consultas SQL analíticas

En esta fase se responden seis preguntas de negocio sobre la bodega de datos GEIH construida en la Fase B. Cada bloque incluye la pregunta, la consulta SQL ejecutada, el resultado tabular y la interpretación correspondiente.

> Antes de ejecutar este notebook, asegúrate de tener el archivo `.env` local con la variable `DATABASE_URL` configurada.

¿Cuál es la distribución de la población en la bodega por sexo, departamento y estado ocupacional?

In [10]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(os.getenv('DATABASE_URL'))

query_1 = text('''
WITH base AS (
    SELECT
        p.sexo,
        u.departamento,
        e.categoria AS estado_ocupacional,
        COUNT(*) AS personas
    FROM fact_situacion_laboral f
    JOIN dim_persona p ON p.id_persona = f.id_persona
    JOIN dim_ubicacion u ON u.id_ubicacion = f.id_ubicacion
    JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
    GROUP BY p.sexo, u.departamento, e.categoria
)
SELECT *
FROM base
ORDER BY departamento, sexo, estado_ocupacional
LIMIT 50;
''')

with engine.connect() as conn:
    df1 = conn.execute(query_1).fetchall()
    print(df1[:10])


[('1', '5', 'Desempleado', 64), ('1', '5', 'Inactivo', 371), ('1', '5', 'Ocupado', 1064), ('2', '5', 'Desempleado', 87), ('2', '5', 'Inactivo', 784), ('2', '5', 'Ocupado', 899), ('1', '8', 'Desempleado', 54), ('1', '8', 'Inactivo', 300), ('1', '8', 'Ocupado', 712), ('2', '8', 'Desempleado', 54)]


## Consulta 2
¿Cuál es el ingreso promedio por sexo y nivel educativo en la población ocupada?

In [11]:
query_2 = text('''
SELECT
    p.sexo,
    p.nivel_educativo,
    CAST(AVG(f.ingreso_laboral) AS NUMERIC(18, 2)) AS ingreso_promedio,
    COUNT(*) AS personas
FROM fact_situacion_laboral f
JOIN dim_persona p ON p.id_persona = f.id_persona
JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
WHERE e.categoria = 'Ocupado'
GROUP BY p.sexo, p.nivel_educativo
ORDER BY p.sexo, ingreso_promedio DESC;
''')

with engine.connect() as conn:
    df2 = conn.execute(query_2).fetchall()
    print(df2[:20])


[('1', '13.0', Decimal('9934285.71'), 50), ('1', '12.0', Decimal('7172176.57'), 274), ('1', '11.0', Decimal('6812431.08'), 543), ('1', '10.0', Decimal('3565044.47'), 2448), ('1', '7.0', Decimal('2943333.33'), 12), ('1', '9.0', Decimal('2545880.06'), 676), ('1', '8.0', Decimal('2216997.88'), 1123), ('1', '6.0', Decimal('1967143.43'), 420), ('1', '5.0', Decimal('1808010.21'), 5128), ('1', '4.0', Decimal('1509026.22'), 2036), ('1', '3.0', Decimal('1399966.33'), 3012), ('1', '2.0', Decimal('1200000.00'), 1), ('1', '1.0', Decimal('968809.70'), 491), ('2', '13.0', Decimal('7618069.88'), 51), ('2', '12.0', Decimal('5903480.77'), 343), ('2', '11.0', Decimal('5180177.66'), 739), ('2', '10.0', Decimal('2810163.72'), 2656), ('2', '7.0', Decimal('2238248.70'), 30), ('2', '9.0', Decimal('2001583.42'), 579), ('2', '8.0', Decimal('1801765.01'), 1539)]


### Interpretación
Este resultado muestra cómo varía el ingreso laboral promedio según el nivel educativo y el sexo dentro de la fuerza laboral ocupada. Los diferenciales sugieren que, en promedio, el mayor capital educativo se asocia con mejor remuneración, aunque la brecha por sexo puede mantenerse según el nivel alcanzado.

## Consulta 3
¿Cuál es la proporción de desempleo por departamento y sexo?

In [12]:
query_3 = text('''
WITH poblacion_total AS (
    SELECT u.departamento, p.sexo, COUNT(*) AS total_personas
    FROM fact_situacion_laboral f
    JOIN dim_ubicacion u ON u.id_ubicacion = f.id_ubicacion
    JOIN dim_persona p ON p.id_persona = f.id_persona
    GROUP BY u.departamento, p.sexo
),
poblacion_desempleada AS (
    SELECT u.departamento, p.sexo, COUNT(*) AS desempleados
    FROM fact_situacion_laboral f
    JOIN dim_ubicacion u ON u.id_ubicacion = f.id_ubicacion
    JOIN dim_persona p ON p.id_persona = f.id_persona
    JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
    WHERE e.categoria = 'Desempleado'
    GROUP BY u.departamento, p.sexo
)
SELECT
    t.departamento,
    t.sexo,
    CAST((COALESCE(d.desempleados, 0) * 100.0) / NULLIF(t.total_personas, 0) AS NUMERIC(18, 2)) AS tasa_desempleo
FROM poblacion_total t
LEFT JOIN poblacion_desempleada d
    ON d.departamento = t.departamento AND d.sexo = t.sexo
ORDER BY tasa_desempleo DESC, t.departamento, t.sexo
LIMIT 20;
''')

with engine.connect() as conn:
    df3 = conn.execute(query_3).fetchall()
    print(df3[:20])


[('San Andrés y Providencia', '1', Decimal('17.08')), ('Putumayo', '1', Decimal('13.64')), ('Vichada', '2', Decimal('13.33')), ('Putumayo', '2', Decimal('12.14')), ('Chocó', '1', Decimal('11.50')), ('Chocó', '2', Decimal('10.49')), ('Vichada', '1', Decimal('10.47')), ('Arauca', '1', Decimal('10.00')), ('San Andrés y Providencia', '2', Decimal('9.87')), ('Sucre', '2', Decimal('9.70')), ('Guaviare', '1', Decimal('9.43')), ('Bolívar', '2', Decimal('9.21')), ('Arauca', '2', Decimal('8.28')), ('Guaviare', '2', Decimal('8.11')), ('Casanare', '2', Decimal('7.55')), ('La Guajira', '2', Decimal('7.51')), ('Boyacá', '1', Decimal('7.33')), ('Tolima', '1', Decimal('7.30')), ('Valle del Cauca', '2', Decimal('6.93')), ('Nariño', '2', Decimal('6.92'))]


### Interpretación
La tasa de desempleo presenta diferencias relevantes entre departamentos y por sexo. Esto permite identificar dónde la precariedad laboral es más pronunciada y si una población específica está más expuesta a la falta de empleo.

## Consulta 4
¿Cuál es el ingreso mensual promedio por hogar según el tipo de vivienda y la calidad del acceso a servicios?

In [13]:
query_4 = text('''
SELECT
    h.tipo_vivienda,
    h.acceso_servicios,
    CAST(AVG(f.ingreso_laboral) AS NUMERIC(18, 2)) AS ingreso_promedio_hogar,
    COUNT(DISTINCT f.id_persona) AS personas_cubiertas
FROM fact_situacion_laboral f
JOIN dim_hogar_vivienda h ON h.id_hogar = f.id_hogar
GROUP BY h.tipo_vivienda, h.acceso_servicios
ORDER BY ingreso_promedio_hogar DESC;
''')

with engine.connect() as conn:
    df4 = conn.execute(query_4).fetchall()
    print(df4[:20])


[('9', 'Completo', Decimal('4510000.00'), 12), ('6', 'Completo', Decimal('4030000.00'), 6), ('1', 'Completo', Decimal('2385166.53'), 36365), ('1', 'Parcial', Decimal('1791161.93'), 12742), ('5', 'Ninguno', Decimal('1705952.38'), 116), ('3', 'Completo', Decimal('1605935.59'), 41), ('2', 'Completo', Decimal('1466304.35'), 40), ('4', 'Completo', Decimal('1344016.67'), 135), ('2', 'Ninguno', Decimal('1300000.00'), 22), ('1', 'Ninguno', Decimal('1297290.32'), 51), ('7', 'Completo', Decimal('1294224.12'), 36), ('2', 'Parcial', Decimal('1246273.06'), 379), ('8', 'Completo', Decimal('1161211.67'), 17), ('5', 'Completo', Decimal('1156437.78'), 153), ('8', 'Ninguno', Decimal('1142181.00'), 23), ('5', 'Parcial', Decimal('1140678.49'), 1828), ('7', 'Parcial', Decimal('1098955.43'), 103), ('4', 'Parcial', Decimal('1063708.39'), 456), ('3', 'Parcial', Decimal('1059201.69'), 200), ('8', 'Parcial', Decimal('1028435.59'), 311)]


### Interpretación
El ingreso promedio por hogar se asocia, como era de esperarse, con mejores condiciones de vivienda y acceso a servicios. Esta relación sugiere que la calidad de la infraestructura y la estructura de la vivienda coinciden con niveles mayores de bienestar económico.

## Consulta 5
¿Hay diferencias en el ingreso laboral por tipo de vivienda y departamento usando una comparación ordenada por rendimiento relativo?

In [14]:
query_5 = text('''
WITH ingreso_departamento AS (
    SELECT
        u.departamento,
        h.tipo_vivienda,
        CAST(AVG(f.ingreso_laboral) AS NUMERIC(18, 2)) AS ingreso_promedio,
        ROW_NUMBER() OVER (
            PARTITION BY u.departamento
            ORDER BY AVG(f.ingreso_laboral) DESC
        ) AS ranking_vivienda
    FROM fact_situacion_laboral f
    JOIN dim_ubicacion u ON u.id_ubicacion = f.id_ubicacion
    JOIN dim_hogar_vivienda h ON h.id_hogar = f.id_hogar
    GROUP BY u.departamento, h.tipo_vivienda
)
SELECT departamento, tipo_vivienda, ingreso_promedio, ranking_vivienda
FROM ingreso_departamento
WHERE ranking_vivienda = 1
ORDER BY ingreso_promedio DESC;
''')

with engine.connect() as conn:
    df5 = conn.execute(query_5).fetchall()
    print(df5[:20])


[('Cundinamarca', '6', None, 1), ('8', '4', None, 1), ('Nariño', '7', None, 1), ('Huila', '3', None, 1), ('Caquetá', '8', None, 1), ('Tolima', '3', None, 1), ('Guainía', '4', None, 1), ('5', '9', Decimal('11566666.67'), 1), ('Valle del Cauca', '3', Decimal('6000000.00'), 1), ('Vaupés', '2', Decimal('4021428.57'), 1), ('Casanare', '1', Decimal('3786004.42'), 1), ('Bogotá D.C.', '1', Decimal('3515939.04'), 1), ('Quindío', '2', Decimal('2800000.00'), 1), ('San Andrés y Providencia', '1', Decimal('2784214.37'), 1), ('Santander', '1', Decimal('2602731.13'), 1), ('Boyacá', '1', Decimal('2530004.86'), 1), ('Caldas', '1', Decimal('2484634.19'), 1), ('Putumayo', '1', Decimal('2425213.68'), 1), ('Risaralda', '1', Decimal('2304524.55'), 1), ('Meta', '1', Decimal('2295702.78'), 1)]


### Interpretación
La consulta prioriza la vivienda asociada al mayor ingreso promedio dentro de cada departamento, lo que permite comparar el contexto residencial más favorable. Esta vista ayuda a detectar si ciertos tipos de vivienda coinciden con mejor desempeño laboral según la zona geográfica.

## Consulta 6
¿En qué medida la migración afecta el ingreso laboral y la permanencia en la fuerza de trabajo?

In [15]:
query_6 = text('''
SELECT
    m.nacio_en_municipio,
    m.razon_migracion,
    e.categoria AS estado_ocupacional,
    CAST(AVG(f.ingreso_laboral) AS NUMERIC(18, 2)) AS ingreso_promedio,
    COUNT(*) AS personas
FROM fact_situacion_laboral f
JOIN dim_migracion m ON m.id_migracion = f.id_migracion
JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
GROUP BY m.nacio_en_municipio, m.razon_migracion, e.categoria
ORDER BY ingreso_promedio DESC
LIMIT 20;
''')

with engine.connect() as conn:
    df6 = conn.execute(query_6).fetchall()
    print(df6[:20])


[('1', '1.0', 'Inactivo', None, 17), ('2', '1.0', 'Desempleado', None, 1), ('3', '3.0', 'Inactivo', None, 11), ('1', None, 'Inactivo', None, 10738), ('3', '9.0', 'Inactivo', None, 27), ('1', '2.0', 'Inactivo', None, 2), ('2', '1.0', 'Inactivo', None, 19), ('1', '3.0', 'Inactivo', None, 3), ('1', None, 'Desempleado', None, 1903), ('2', '3.0', 'Inactivo', None, 1), ('2', '2.0', 'Inactivo', None, 1), ('3', '1.0', 'Inactivo', None, 510), ('2', None, 'Desempleado', None, 1082), ('1', '9.0', 'Inactivo', None, 2), ('3', '9.0', 'Desempleado', None, 2), ('3', '2.0', 'Inactivo', None, 44), ('3', '2.0', 'Desempleado', None, 11), ('2', None, 'Inactivo', None, 9200), ('3', '1.0', 'Desempleado', None, 93), ('2', '9.0', 'Inactivo', None, 5)]


### Interpretación
La migración parece relacionarse con diferencias en el ingreso promedio y la categoría ocupacional, lo que sugiere que la trayectoria migratoria puede influir en la participación laboral. Aunque el resultado no prueba causalidad, sí ofrece una señal útil para profundizar en segmentaciones por movilidad y tipo de empleo.